In [0]:
from pyspark.sql.functions import *

catalog = "bike_data"
bronze_schema = "bronze"
silver_schema = "silver"

print("=" * 80)
print("SILVER TRANSFORMATION: erp_px_cat_g1v2")
print("=" * 80)

# Section 1: Read Bronze table
print("\nSection 1: Reading Bronze table")
df = spark.table(f"{catalog}.{bronze_schema}.erp_px_cat_g1v2")
initial_count = df.count()
print(f"Initial rows: {initial_count}")

# Section 2: Transform data
print("\nSection 2: Data transformation")

# 2.1 Remove duplicates
print("  2.1 Removing duplicates by ID")
df = df.dropDuplicates(subset=["ID"])
print(f"  Rows after dedup: {df.count()}")

# 2.2 Clean strings
print("  2.2 Cleaning string columns")
df = df.withColumn("CAT", trim(upper(col("CAT"))))
df = df.withColumn("SUBCAT", trim(upper(col("SUBCAT"))))
df = df.withColumn("MAINTENANCE", trim(upper(col("MAINTENANCE"))))

# 2.3 Remove invalid rows
print("  2.3 Removing invalid rows")
df = df.filter(col("ID").isNotNull())

print(f"  Rows after transformations: {df.count()}")

# Section 3: Sanity checks
print("\nSection 3: Sanity checks")
null_check = df.select([count(when(col(c).isNull(), c)).alias(c) for c in df.columns])
display(null_check)
display(df.limit(3))

# Section 4: Write to Silver
print("\nSection 4: Writing to Silver table")
silver_table = f"{catalog}.{silver_schema}.categories"
df.write.mode("overwrite").format("delta").saveAsTable(silver_table)

final_count = df.count()
print(f"Written to: {silver_table}")
print(f"Final rows: {final_count}")
print(f"Removed: {initial_count - final_count} rows")